# RUL Prediction — Phase 1, Notebook 4
## C-MAPSS FD001: LSTM Model — Training, Evaluation & Comparison

**Dissertation:** Deep Learning-Based Remaining Useful Life Prediction for Industrial Rotating Equipment
**Author:** K Vatsav Sai Ram · BITS ID 2024TM05021

---

### The core of the dissertation
Notebook 3 set the baselines to beat (Random Forest RMSE 16.24, XGBoost RMSE 12.82). Now we build the
**LSTM** — a recurrent network that reads each 30-cycle window *as a sequence*, using the time-ordering
the tree baselines discarded.

This notebook:
1. Builds an LSTM regression model.
2. Trains it correctly (see the **target-scaling** section — this matters a lot).
3. Evaluates with **RMSE** and the **PHM score**, same as the baselines.
4. Compares all three models — the result table your mid-semester report is built around.

> **Before running:** set a GPU runtime — *Runtime -> Change runtime type -> T4 GPU*.


## 1. Setup & load data

We load the same `.npz` arrays and reuse the identical metric definitions, so every model is judged on
exactly the same footing.

In [ ]:
import numpy as np, os, json
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)
plt.rcParams["figure.figsize"] = (10, 4)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU') or "NONE - set Runtime > GPU for speed")

# --- project paths: Colab+Drive if available, else a local folder ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/RUL_Dissertation"
except ImportError:
    PROJECT_DIR = os.path.abspath(".")

DATA_DIR     = os.path.join(PROJECT_DIR, "data", "cmapss")
ARTIFACT_DIR = os.path.join(PROJECT_DIR, "artifacts")
PROC_DIR     = os.path.join(PROJECT_DIR, "processed")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
for _d in (ARTIFACT_DIR, PROC_DIR, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)
assert os.path.isdir(DATA_DIR), f"Place the C-MAPSS .txt files in {DATA_DIR}"
print("DATA_DIR:", DATA_DIR)

d = np.load(os.path.join(PROC_DIR, "fd001_windows.npz"))
X_train, y_train, X_test, y_test = d["X_train"], d["y_train"], d["X_test"], d["y_test"]
WINDOW, N_FEAT = int(d["window"]), int(d["n_features"])

with open(os.path.join(ARTIFACT_DIR, "config_FD001.json")) as f:
    cfg = json.load(f)
RUL_CAP = cfg["rul_cap"]          # 125

print("X_train", X_train.shape, "| X_test", X_test.shape, "| RUL cap:", RUL_CAP)

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def phm_score(y_true, y_pred):
    dd = y_pred - y_true
    return float(np.sum(np.where(dd < 0, np.exp(-dd/13) - 1, np.exp(dd/10) - 1)))

## 2. Reference points: what does "doing nothing" score?

Before training, it is worth knowing what a **trivial constant predictor** scores. If a trained model
lands near this, it has learned nothing — and we would otherwise not notice, because an RMSE number
on its own looks like a result.

This check is what exposed a serious training failure in the first version of this notebook
(see Section 4), so we keep it in permanently.

In [ ]:
mean_rul = float(y_train.mean())
const_pred = np.full_like(y_test, mean_rul, dtype=float)
print(f"Training RUL mean: {mean_rul:.1f}")
print(f"CONSTANT predictor (always predicts {mean_rul:.0f}):")
print(f"   RMSE = {rmse(y_test, const_pred):.2f}   PHM = {phm_score(y_test, const_pred):.0f}")
print("\n--> Any model scoring near this has collapsed to the mean and learned nothing.")

## 3. A validation split for early stopping

We hold out 15% of the *training* windows as a validation set. This lets us watch for overfitting and
stop training at the right time — the **test set stays untouched** until final scoring.

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr_raw, y_val_raw = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED, shuffle=True
)
print("train windows:", X_tr.shape[0], "| val windows:", X_val.shape[0])

## 4. Target scaling — the step that makes or breaks this model

**This is the most important cell in the notebook.**

Our inputs are min-max scaled to `[0, 1]`, but the target RUL is on a `0–125` scale. Training directly
on that raw target fails badly:

- The initial MSE equals the variance of `y` (≈ 41.8² ≈ 1750), so the loss surface is dominated by
  scale rather than signal.
- The optimiser slides into the trivial "just predict the mean" solution — the model outputs the **same
  constant for every engine** (prediction std = 0).
- Validation loss then plateaus at that constant, so **early stopping halts training**, locking in the
  failure.

Measured on this exact data:

| Target handling | RMSE | PHM score | Prediction spread |
|---|---|---|---|
| Raw `y` (0–125) | **40.36** | **16102** | constant 79.2 (std **0.0**) — collapsed |
| Scaled `y/125` | **13.48** | **295** | 8 → 123 (std 38.8) — healthy |

So we **scale the target to `[0, 1]` for training and scale the predictions back by `RUL_CAP`
afterwards.** Same architecture, same data — this one step takes RMSE from 40 to 13.

*(This is a genuinely reportable methodological finding for your dissertation: input normalisation is
routinely discussed in the literature, but target normalisation is what actually decided whether this
model trained at all.)*

In [ ]:
# Scale targets to [0, 1] for training; predictions get scaled back by RUL_CAP later.
y_tr  = y_tr_raw  / RUL_CAP
y_val = y_val_raw / RUL_CAP

print(f"Raw target range   : {y_tr_raw.min():.0f} to {y_tr_raw.max():.0f}")
print(f"Scaled target range: {y_tr.min():.3f} to {y_tr.max():.3f}")

## 5. The LSTM architecture

A compact, proven design for C-MAPSS RUL:

- **LSTM(64, return_sequences=True)** — reads the sequence, passes the full sequence on.
- **Dropout(0.2)** — regularisation against overfitting.
- **LSTM(32)** — condenses to a single vector summarising the window.
- **Dropout(0.2)**
- **Dense(16, relu)** -> **Dense(1)** — regresses to a single (scaled) RUL value.

Loss is **MSE**; we also track **MAE**. Kept intentionally modest (~33k parameters) so it trains fast
and doesn't overfit FD001's 100 engines.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input
from tensorflow.keras.optimizers import Adam

def build_lstm(window, n_feat):
    model = Sequential([
        Input(shape=(window, n_feat)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer=Adam(1e-3), loss="mse", metrics=["mae"])
    return model

tf.keras.utils.set_random_seed(SEED)
model = build_lstm(WINDOW, N_FEAT)
model.summary()

## 6. Train

Three callbacks:
- **EarlyStopping** (patience 25) — stop when validation loss genuinely stops improving, and restore the
  best weights. The generous patience matters: with patience 12 the first version stopped at epoch 20,
  before the model had escaped the mean-prediction plateau.
- **ReduceLROnPlateau** — halve the learning rate when progress stalls, which lets the model keep
  refining instead of bouncing around a minimum.
- **ModelCheckpoint** — save the best model to Drive.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

ckpt_path = os.path.join(MODEL_DIR, "lstm_fd001.keras")
callbacks = [
    EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5),
    ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True),
]

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=200, batch_size=256,
    callbacks=callbacks, verbose=1,
)
print("\nEpochs run:", len(history.history["loss"]))
print("Best model saved to:", ckpt_path)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(history.history["loss"], label="train loss (MSE, scaled target)")
plt.plot(history.history["val_loss"], label="val loss (MSE, scaled target)")
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.title("LSTM training curves"); plt.legend()
plt.tight_layout(); plt.show()

## 7. Predict and un-scale

The model outputs a value in `[0, 1]`, so we multiply by `RUL_CAP` to get cycles back.

In [ ]:
lstm_pred = model.predict(X_test, verbose=0).ravel() * RUL_CAP
lstm_pred = np.clip(lstm_pred, 0, None)

lstm_rmse = rmse(y_test, lstm_pred)
lstm_phm  = phm_score(y_test, lstm_pred)

print(f"Prediction spread: min={lstm_pred.min():.1f}  max={lstm_pred.max():.1f}  std={lstm_pred.std():.1f}")
print(f"LSTM  ->  RMSE = {lstm_rmse:.2f}   PHM score = {lstm_phm:.1f}")

### 7.1 Collapse guard

An automatic check that the model is actually discriminating between engines rather than emitting a
constant. This is the safety net for the failure mode described in Section 4.

In [ ]:
if lstm_pred.std() < 5:
    print("*** WARNING: predictions are nearly constant (std < 5) ***")
    print("    The model has collapsed to the mean and has NOT learned.")
    print("    Check that the target was scaled (Section 4) before training.")
else:
    print(f"OK - predictions vary properly (std = {lstm_pred.std():.1f}).")
    print(f"     Constant-predictor RMSE was {rmse(y_test, const_pred):.2f}; "
          f"this model is {rmse(y_test, const_pred) - lstm_rmse:.1f} better.")

In [ ]:
order = np.argsort(y_test)
plt.figure(figsize=(11,4))
plt.plot(y_test[order], "o-", color="#10243B", ms=3, label="true RUL")
plt.plot(lstm_pred[order], "o", color="#0E7C7B", ms=4, alpha=0.75, label="LSTM predicted")
plt.xlabel("test engine (sorted by true RUL)"); plt.ylabel("RUL (cycles)")
plt.title(f"LSTM predictions on FD001 test set  (RMSE={lstm_rmse:.2f})"); plt.legend()
plt.tight_layout(); plt.show()

## 8. Final comparison — the mid-sem headline result

Bring the baselines (saved in Notebook 3) alongside the LSTM. This table is the central result of the
Phase-1 work and of your mid-semester report.

In [ ]:
import pandas as pd
base = pd.read_csv(os.path.join(ARTIFACT_DIR, "baseline_results_FD001.csv"))
lstm_row = pd.DataFrame({"Model": ["LSTM"], "RMSE": [round(lstm_rmse,2)], "PHM_Score": [round(lstm_phm,1)]})
comparison = pd.concat([base, lstm_row], ignore_index=True).sort_values("RMSE").reset_index(drop=True)
print(comparison.to_string(index=False))
comparison.to_csv(os.path.join(ARTIFACT_DIR, "model_comparison_FD001.csv"), index=False)

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].bar(comparison["Model"], comparison["RMSE"], color="#0E7C7B"); ax[0].set_title("RMSE (lower better)")
ax[1].bar(comparison["Model"], comparison["PHM_Score"], color="#10243B"); ax[1].set_title("PHM Score (lower better)")
for a in ax: a.tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

## 9. Summary — Phase 1 complete (mid-sem deliverable)

**What Phase 1 delivered:**
- Full C-MAPSS FD001 pipeline: load -> EDA -> preprocess -> baselines -> LSTM.
- Metrics: **RMSE** and the asymmetric **PHM score** (encoding "late predictions cost more").
- A clean model comparison (RF vs XGBoost vs LSTM) saved to Drive.
- A documented **methodological finding**: target scaling is decisive for LSTM convergence here
  (RMSE 40.4 -> 13.5), plus a permanent collapse guard so the failure cannot recur unnoticed.
- Trained LSTM saved for reuse (SHAP explainability comes next).

**Expected numbers on FD001:** LSTM RMSE ≈ 13.5 (PHM ≈ 295), Random Forest ≈ 16.2, XGBoost ≈ 12.8.
The LSTM comfortably beats Random Forest; XGBoost is competitive on this subset.

**How to report the XGBoost comparison honestly:** FD001 is the *simplest* subset — one operating
condition, one fault mode — which is exactly where sequence modelling has least to offer, so a strong
gradient-boosting model on flattened windows is expected to be competitive. The LSTM's advantage should
widen on **FD002/FD004** (six operating conditions, two fault modes), where temporal context matters
much more. Stating this plainly, and then testing it, is a stronger dissertation than forcing the LSTM
to win on the easiest subset.

**After mid-sem:** SHAP explainability (which sensors/timesteps drive the RUL), extend to FD002-FD004,
then Phase 2 — apply the framework to the RBT roller data.
